# Spatial + Classification (Molmo)

This notebook runs **spatial localization** and **accident type classification** using `MolmoReasoner`.

Before running:
- Generate temporal predictions first in `../temporal` (needed: `../temporal/results/temporal_pred.csv`).
- Install dependencies from `../../requirements.txt`.
- Run from this folder so relative paths resolve correctly.

Main output:
- `./results/molmo_oracle_analysis.pkl`
- `./results/molmo_pred_analysis.pkl`

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dataset.accident_dataset import get_dataset_paths, resolve_dataset_root

sys.path.append('../..') # Path to reasoning folder
from reasoning.qwen import QwenVLReasoner
from reasoning.molmo import MolmoReasoner
from reasoning.utils import get_frame_by_id, get_every_nth_frame, crop_with_bbox, crop_around_point
from reasoning.visualize import visualize_points_on_image


def run_analysis(model, df, model_name, col_frame='accident_frame'):
    cls_prompt1 = (
        "What type of traffic accident is shown? "
        "Choose one from: "
        "t-bone - One vehicle crashes perpendicularly into the side of another."
        "sideswipe - Two vehicles traveling scrape along each other’s sides."
        "rear-end - One vehicle collides with the back of another."
        "single - A crash involving only one vehicle, with no other vehicles hit."
        "head-on - Two vehicles traveling in opposite directions collide front to front."
        "Return only single word"
    )
    
    cls_prompt2 = (
        "What type of traffic accident is shown? "
        "Choose from: t-bone, sideswipe, rear-end, single-vehicle, head-on. Return only single word"
    )
    
    
    results = []
    for i, row in tqdm(df.iterrows(), total=len(df)):
        with open(f"progress_{model_name}.log", "a") as f:  # <-- 'a' append mode
            print(f"Processed row {i+1}/{len(df)}", file=f)
        data = row.to_dict()
    
        img_gt = get_frame_by_id(row['video_path'], row[col_frame])
    
        # Classification
        crop_point = crop_around_point(
            img_gt,
            point=(row['center_x']*row['width'], row['center_y']*row['height']),
            crop_percent=0.3
        )
        crop_bbox = crop_with_bbox(
            img_gt,
            bbox=(row['x1']*row['width'], row['y1']*row['height'], row['x2']*row['width'], row['y2']*row['height']),
            padding_percent=0.3
        )
        crop_full =  img_gt
    
        imgs = {'crop_point': crop_point, 'crop_bbox': crop_bbox, 'crop_full': crop_full}
        for name, img_oracle in imgs.items():
            outputs = {}
            
            label, raw_text = model.accident_cause_reasoning(img_oracle, prompt=cls_prompt1)
            outputs['cls_prompt1_label'] = label
            outputs['cls_prompt1_raw_text'] = raw_text
    
            label, raw_text = model.accident_cause_reasoning(img_oracle, prompt=cls_prompt2)
            outputs['cls_prompt2_label'] = label
            outputs['cls_prompt2_raw_text'] = raw_text
    
            data[name] = outputs

        # Spatial reasoning
        (x, y), spatial_raw = model.accident_spatial_reasoning(img_gt)
        data['spatial'] = {
            'x': x,
            'y': y,
            'raw_text': spatial_raw,
        }
        results.append(data)

    os.makedirs(f'results/', exist_ok=True)
    torch.save(results, f'results/{model_name}_analysis.pkl')


dataset_path = Path('../../dataset')
dataset_root = resolve_dataset_root(dataset_path)
_, metadata_path = get_dataset_paths(
    dataset_root, kind="real"
)
df = pd.read_csv(metadata_path)
df["video_path"] = df["path"].apply(lambda x: os.path.join(dataset_root, x))

df_pred = pd.read_csv('../temporal/results/temporal_pred.csv')
df['pred_frame_qwen'] = df_pred['pred_frame_qwen']
df['pred_frame_molmo'] = df_pred['pred_frame_molmo']

In [ ]:
model = MolmoReasoner()
name = 'molmo'

# Run molmo spatial and classification reasoning with temporal oracle
run_analysis(model, df=df, model_name=name + '_oracle', col_frame='accident_frame')

# Run molmo spatial and classification reasoning with molmo temporal prediction
run_analysis(model, df=df, model_name=name + '_pred', col_frame='pred_frame_molmo')